# AKI ROC Analysis

**Paper:** *Predictive Value of Hydration Markers for Acute Kidney Injury Risk in Young vs. Older Females*

## Overview

This notebook reproduces all statistics reported in the manuscript, including:

- **Table 1** — Participant characteristics (age, BMI) with Mann-Whitney U tests and rank-biserial correlation effect sizes
- **Aim 1** — ROC analysis: AUC, 95% CI, and Youden's J for each hydration marker by age group (unpaired DeLong tests)
- **Aim 2** — Paired DeLong comparison of spot USG vs. 24-hour USG
- **Figure 1** — AUC dot plot with 95% CI by age group
- **Figure 2** — Paired ROC curves for spot vs. 24-hour USG

## Data

Place the input file `AKI_data.csv` in the same directory as this notebook before running.

**Expected columns:** `Age Group`, `ID`, `Condition`, `IGFBP7*TIMP-2 ((ng/mL)^2)/1000`, `AKI Risk`, `percent change weight`, `Spot USG`, `24hr USG`, `24hr Osmo`, `Plasma Osmo`, `Screening BMI`, `Age`

**Note on missing data:** Two participants (AH13 and SRWR08) are missing `AKI Risk` values and are excluded from ROC analyses but included in Table 1 demographics.

## Requirements

- R ≥ 4.0
- Packages: `pROC`, `ggplot2`, `dplyr`, `zoo`, `coin`, `rstatix`
- IRkernel (for Jupyter)

Run **Cell 1** once to install any missing packages.

In [ ]:
# ── Cell 1: Install missing packages (run once) ───────────────────────────────
pkgs <- c("pROC", "ggplot2", "dplyr", "zoo", "coin", "rstatix")
install.packages(pkgs[!pkgs %in% installed.packages()[, "Package"]])

In [ ]:
# ── Cell 2: Load libraries ────────────────────────────────────────────────────
library(pROC)
library(ggplot2)
library(dplyr)
library(coin)
library(rstatix)
cat("Libraries loaded.\n")

In [ ]:
# ── Cell 3: Load data ─────────────────────────────────────────────────────────
# Place AKI_data.csv in the same folder as this notebook.
# If running from a different directory, update the path below.
data_file <- "AKI_data.csv"

df_raw <- read.csv(data_file, na.strings = c("", "NA", "NaN"))

# Standardise first column name (handles BOM character from Excel exports)
colnames(df_raw)[1] <- "Age_Group"

cat("Columns:\n"); print(colnames(df_raw))
cat("Age_Group values:", paste(unique(df_raw$Age_Group), collapse = ", "), "\n")

# Keep full dataset for Table 1 demographics (retains participants with missing AKI Risk)
df_demo <- df_raw

# ROC analysis dataset: drop rows with missing AKI Risk
df_raw        <- df_raw[!is.na(df_raw$AKI.Risk), ]
df_raw$AKI_binary <- ifelse(tolower(df_raw$AKI.Risk) == "yes", 1, 0)

cat("\nObservations after dropping missing AKI Risk:", nrow(df_raw), "\n")
cat("\nAKI distribution by group:\n")
print(table(df_raw$Age_Group, df_raw$AKI_binary,
            dnn = c("Age_Group", "AKI (0 = no, 1 = yes)")))

In [ ]:
# ── Cell 4: Define markers and check missing values ───────────────────────────
# R converts column name spaces to dots on import:
#   'Spot USG'             -> Spot.USG
#   '24hr USG'             -> X24hr.USG
#   '24hr Osmo'            -> X24hr.Osmo
#   'Plasma Osmo'          -> Plasma.Osmo
#   'percent change weight'-> percent.change.weight

markers       <- c("Spot.USG", "X24hr.USG", "X24hr.Osmo", "Plasma.Osmo", "percent.change.weight")
marker_labels <- c("Spot USG", "24-hr USG", "24-hr Osmo", "Plasma Osmo", "% Body Mass Change")

cat("=== Missing values (n = 50 obs with known AKI Risk) ===\n")
for (i in seq_along(markers))
  cat(sprintf("  %-22s  missing = %d  |  available = %d\n",
              marker_labels[i],
              sum(is.na(df_raw[[markers[i]]])),
              sum(!is.na(df_raw[[markers[i]]]))))

---
## Table 1 — Participant Characteristics

Demographics are drawn from `df_demo` (all 26 participants), not the ROC dataset, so participants with missing AKI Risk data are correctly included here.

Statistics: Mann-Whitney U test, median [IQR], rank-biserial correlation (rbc = 1 − 2W / (n₁·n₂)).

In [ ]:
# ── Cell 5: Table 1 ───────────────────────────────────────────────────────────
demo       <- df_demo[!duplicated(df_demo$ID), ]   # one row per participant
young_demo <- demo[demo$Age_Group == "Young", ]
older_demo <- demo[demo$Age_Group == "Older", ]

cat(sprintf("Unique participants: YF n = %d  |  OF n = %d\n\n",
            nrow(young_demo), nrow(older_demo)))

# Helper: prints median [IQR], Mann-Whitney U p-value, and rank-biserial correlation
table1_var <- function(var_name, label, y_df, o_df) {
  y   <- na.omit(y_df[[var_name]])
  o   <- na.omit(o_df[[var_name]])
  wt  <- wilcox.test(y, o, exact = FALSE)
  rbc <- 1 - (2 * as.numeric(wt$statistic)) / (length(y) * length(o))
  cat(sprintf("%s:\n  YF: %.1f [%.1f]   OF: %.1f [%.1f]   p = %.4f   rbc = %.2f\n\n",
              label, median(y), IQR(y), median(o), IQR(o), wt$p.value, rbc))
}

cat("=== TABLE 1 ===\n\n")
table1_var("Age",            "Age (years)", young_demo, older_demo)
table1_var("Screening.BMI", "BMI (kg/m²)", young_demo, older_demo)

---
## Helper functions

Run this cell before Aims 1 and 2.

In [ ]:
# ── Cell 6: Helper functions ──────────────────────────────────────────────────

# Build a ROC object safely; returns NULL if insufficient data
safe_roc <- function(response, predictor, direction = "auto") {
  ok <- complete.cases(response, predictor)
  if (sum(ok) < 5 || length(unique(response[ok])) < 2) return(NULL)
  tryCatch(roc(response[ok], predictor[ok], quiet = TRUE, direction = direction),
           error = function(e) NULL)
}

# Extract AUC, 95% CI, Youden's J, and optimal threshold from a ROC object
roc_summary <- function(roc_obj) {
  if (is.null(roc_obj)) return(NULL)
  auc_val <- as.numeric(auc(roc_obj))
  ci_obj  <- tryCatch(ci.auc(roc_obj, conf.level = 0.95), error = function(e) NULL)
  sens <- rev(roc_obj$sensitivities)
  spec <- rev(roc_obj$specificities)
  thre <- rev(roc_obj$thresholds)
  J    <- sens + spec - 1
  best <- which.max(J)
  if (length(best) == 0) best <- 1L
  list(
    auc       = auc_val,
    ci_lo     = if (!is.null(ci_obj)) as.numeric(ci_obj[1]) else NA_real_,
    ci_hi     = if (!is.null(ci_obj)) as.numeric(ci_obj[3]) else NA_real_,
    J         = J[best],
    threshold = thre[best],
    sens      = sens[best],
    spec      = spec[best],
    n         = length(roc_obj$cases) + length(roc_obj$controls)
  )
}

# Safely extract a single scalar (returns NA if length 0 or NULL)
s1 <- function(x) if (length(x) == 1 && !is.null(x)) x else NA_real_

# DeLong test — unpaired (comparing two independent ROC curves)
delong_unpaired <- function(r1, r2) {
  if (is.null(r1) || is.null(r2)) return(list(p = NA_real_, z = NA_real_))
  tryCatch({
    t <- roc.test(r1, r2, method = "delong", paired = FALSE)
    list(p = t$p.value, z = as.numeric(t$statistic))
  }, error = function(e) list(p = NA_real_, z = NA_real_))
}

# DeLong test — paired (comparing two ROC curves on the same subjects)
delong_paired <- function(r1, r2) {
  if (is.null(r1) || is.null(r2)) return(list(p = NA_real_, z = NA_real_))
  tryCatch({
    t <- roc.test(r1, r2, method = "delong", paired = TRUE)
    list(p = t$p.value, z = as.numeric(t$statistic))
  }, error = function(e) list(p = NA_real_, z = NA_real_))
}

cat("Helper functions ready.\n")

---
## Aim 1 — Diagnostic accuracy by age group

For each of the five hydration markers, ROC analysis was performed separately in young females (YF) and older females (OF). AUC, 95% CI, Youden's J index, and optimal threshold are reported. Group differences were assessed with unpaired DeLong tests.

In [ ]:
# ── Cell 7: Aim 1 ─────────────────────────────────────────────────────────────
young <- df_raw[df_raw$Age_Group == "Young", ]
older <- df_raw[df_raw$Age_Group == "Older", ]
cat(sprintf("Observations — Young: %d  |  Older: %d\n\n", nrow(young), nrow(older)))

aim1_results <- data.frame()

for (i in seq_along(markers)) {
  m   <- markers[i]
  lab <- marker_labels[i]

  ry <- safe_roc(young$AKI_binary, young[[m]])
  ro <- safe_roc(older$AKI_binary, older[[m]])

  # Ensure both curves use the same direction before comparing
  if (!is.null(ry) && !is.null(ro) && ry$direction != ro$direction)
    ro <- safe_roc(older$AKI_binary, older[[m]], direction = ry$direction)

  sy <- roc_summary(ry)
  so <- roc_summary(ro)
  dl <- delong_unpaired(ry, ro)

  cat(sprintf("--- %s ---\n", lab))
  if (!is.null(sy))
    cat(sprintf("  YF (n = %d): AUC = %.2f (95%% CI %.2f–%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
                sy$n, sy$auc, sy$ci_lo, sy$ci_hi, sy$J, sy$threshold, sy$sens, sy$spec))
  if (!is.null(so))
    cat(sprintf("  OF (n = %d): AUC = %.2f (95%% CI %.2f–%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
                so$n, so$auc, so$ci_lo, so$ci_hi, so$J, so$threshold, so$sens, so$spec))
  cat(sprintf("  DeLong (unpaired): Z = %.3f  p = %.4f\n\n",
              ifelse(is.na(dl$z), 0, dl$z),
              ifelse(is.na(dl$p), 1, dl$p)))

  aim1_results <- bind_rows(aim1_results,
    data.frame(marker = lab, group = "YF",
               auc = s1(sy$auc), ci_lo = s1(sy$ci_lo), ci_hi = s1(sy$ci_hi),
               J = s1(sy$J), threshold = s1(sy$threshold),
               sens = s1(sy$sens), spec = s1(sy$spec), delong_p = s1(dl$p)),
    data.frame(marker = lab, group = "OF",
               auc = s1(so$auc), ci_lo = s1(so$ci_lo), ci_hi = s1(so$ci_hi),
               J = s1(so$J), threshold = s1(so$threshold),
               sens = s1(so$sens), spec = s1(so$spec), delong_p = s1(dl$p))
  )
}

---
## Aim 2 — Spot USG vs. 24-hour USG (paired DeLong)

The paired DeLong test compares the diagnostic accuracy of spot USG and 24-hour USG on the subset of participants with complete data for both markers (n = 45).

In [ ]:
# ── Cell 8: Aim 2 ─────────────────────────────────────────────────────────────
ok   <- complete.cases(df_raw$Spot.USG, df_raw$X24hr.USG, df_raw$AKI_binary)
df_c <- df_raw[ok, ]
cat(sprintf("Complete cases for paired test: %d\n\n", nrow(df_c)))

roc_spot <- safe_roc(df_c$AKI_binary, df_c$Spot.USG)
roc_24hr <- safe_roc(df_c$AKI_binary, df_c$X24hr.USG)
s_spot   <- roc_summary(roc_spot)
s_24hr   <- roc_summary(roc_24hr)
dl_aim2  <- delong_paired(roc_spot, roc_24hr)

cat("Spot USG:\n")
cat(sprintf("  AUC = %.2f (95%% CI %.2f–%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
            s_spot$auc, s_spot$ci_lo, s_spot$ci_hi,
            s_spot$J, s_spot$threshold, s_spot$sens, s_spot$spec))
cat("\n24-hr USG:\n")
cat(sprintf("  AUC = %.2f (95%% CI %.2f–%.2f)  J = %.3f  threshold > %.4f  Sens = %.2f  Spec = %.2f\n",
            s_24hr$auc, s_24hr$ci_lo, s_24hr$ci_hi,
            s_24hr$J, s_24hr$threshold, s_24hr$sens, s_24hr$spec))
cat(sprintf("\nPaired DeLong (Spot vs. 24-hr USG): Z = %.3f  p = %.4f\n", dl_aim2$z, dl_aim2$p))

---
## Figure 1 — AUC dot plot with 95% CI

AUC values and 95% confidence intervals for each hydration marker, stratified by age group. DeLong p-values for group comparisons are shown at right.

In [ ]:
# ── Cell 9: Figure 1 ──────────────────────────────────────────────────────────
options(repr.plot.width = 11, repr.plot.height = 7)

COL_YOUNG <- "#5CB8B2"   # teal — young females
COL_OLDER <- "#FFC72C"   # gold  — older females
COL_TEXT  <- "#1a1a1a"

dot_data        <- aim1_results
dot_data$marker <- factor(dot_data$marker, levels = rev(marker_labels))
dot_data$group  <- factor(dot_data$group,  levels = c("YF", "OF"))

annot <- dot_data %>%
  group_by(marker) %>%
  summarise(delong_p = first(delong_p), .groups = "drop") %>%
  mutate(p_label = ifelse(delong_p < 0.05,
                          sprintf("p = %.3f *", delong_p),
                          sprintf("p = %.2f",   delong_p)))

theme_dot <- theme_classic(base_size = 18) +
  theme(
    text               = element_text(color = COL_TEXT),
    axis.title         = element_text(size = 20),
    axis.text          = element_text(size = 16, color = COL_TEXT),
    axis.line          = element_line(linewidth = 1.5, color = COL_TEXT),
    axis.ticks         = element_line(linewidth = 1.2, color = COL_TEXT),
    legend.position    = "top",
    legend.title       = element_blank(),
    legend.text        = element_text(size = 16),
    plot.title         = element_text(size = 18, face = "bold", hjust = 0.5),
    panel.grid.major.x = element_line(color = "grey90", linewidth = 0.8)
  )

p_dot <- ggplot(dot_data, aes(x = auc, y = marker, color = group)) +
  geom_vline(xintercept = 0.5, linetype = "dashed", color = "grey60", linewidth = 1.0) +
  geom_errorbar(aes(xmin = ci_lo, xmax = ci_hi),
                width = 0.2, linewidth = 1.0,
                position = position_dodge(width = 0.5),
                orientation = "y") +
  geom_point(size = 4.5, position = position_dodge(width = 0.5)) +
  geom_text(data = annot, aes(x = 1.02, y = marker, label = p_label),
            color = "grey30", size = 4.5, hjust = 0, inherit.aes = FALSE) +
  scale_color_manual(values  = c(YF = COL_YOUNG, OF = COL_OLDER),
                     labels  = c(YF = "Young females", OF = "Older females")) +
  scale_x_continuous(name   = "Area Under the Curve (AUC)",
                     limits = c(0.0, 1.18),
                     breaks = c(0.25, 0.5, 0.75, 1.0),
                     labels = c("0.25", "0.50", "0.75", "1.00")) +
  scale_y_discrete(name = NULL) +
  labs(title = "AUC by Hydration Marker and Age Group") +
  theme_dot

dir.create("figures", showWarnings = FALSE)
ggsave("figures/Fig1_AUC_dotplot.png", p_dot, width = 11, height = 7, dpi = 300, bg = "white")
cat("Saved: figures/Fig1_AUC_dotplot.png\n")
p_dot

---
## Figure 2 — Paired ROC curves: Spot vs. 24-hour USG

ROC curves for spot USG and 24-hour USG with bootstrap 95% confidence bands (200 resamples). The paired DeLong p-value is shown in the lower right.

In [ ]:
# ── Cell 10: Figure 2 ─────────────────────────────────────────────────────────
options(repr.plot.width = 9, repr.plot.height = 8)

COL_SPOT <- "#A6192E"   # dark red — spot USG
COL_24HR <- "#FFC72C"   # gold    — 24-hr USG
COL_CI   <- "#aaaaaa"
COL_DIAG <- "grey55"

# Convert ROC object to data frame for ggplot
roc_to_df <- function(roc_obj, label) {
  if (is.null(roc_obj)) return(NULL)
  data.frame(
    fpr   = 1 - rev(roc_obj$specificities),
    tpr   = rev(roc_obj$sensitivities),
    label = label
  )
}

# Bootstrap confidence band for a ROC curve
ci_band <- function(roc_obj, n_boot = 200) {
  if (is.null(roc_obj)) return(NULL)
  ci_obj <- tryCatch(
    ci.se(roc_obj, specificities = seq(0, 1, 0.01),
          conf.level = 0.95, method = "bootstrap",
          boot.n = n_boot, progress = "none"),
    error = function(e) NULL)
  if (is.null(ci_obj)) return(NULL)
  data.frame(
    fpr   = 1 - as.numeric(rownames(ci_obj)),
    lower = as.numeric(ci_obj[, 1]),
    upper = as.numeric(ci_obj[, 3])
  )
}

# Format p-value for annotation
p_fmt <- function(p) {
  if (is.na(p))    return("p = NA")
  if (p < 0.001)   return("p < 0.001")
  sprintf("p = %.3f", p)
}

theme_roc <- theme_classic(base_size = 20) +
  theme(
    text            = element_text(color = COL_TEXT),
    axis.title      = element_text(size = 22),
    axis.text       = element_text(size = 18, color = COL_TEXT),
    axis.line       = element_line(linewidth = 1.8, color = COL_TEXT),
    axis.ticks      = element_line(linewidth = 1.4, color = COL_TEXT),
    plot.title      = element_text(size = 20, face = "bold", hjust = 0.5),
    legend.position = "none"
  )

# Build data objects (roc_spot, roc_24hr, df_c defined in Cell 8)
ci_spot_band <- ci_band(roc_spot)
ci_24hr_band <- ci_band(roc_24hr)
df_spot      <- roc_to_df(roc_spot, "spot")
df_24        <- roc_to_df(roc_24hr, "24hr")
ci_spot_auc  <- ci.auc(roc_spot)
ci_24hr_auc  <- ci.auc(roc_24hr)

label_spot <- sprintf("Spot USG\nAUC = %.2f (%.2f\u2013%.2f)",
                      as.numeric(auc(roc_spot)),
                      as.numeric(ci_spot_auc[1]), as.numeric(ci_spot_auc[3]))
label_24hr <- sprintf("24-hr USG\nAUC = %.2f (%.2f\u2013%.2f)",
                      as.numeric(auc(roc_24hr)),
                      as.numeric(ci_24hr_auc[1]), as.numeric(ci_24hr_auc[3]))

p_roc <- ggplot(df_24, aes(x = fpr, y = tpr)) +
  { if (!is.null(ci_24hr_band))
      geom_ribbon(data = ci_24hr_band, aes(x = fpr, ymin = lower, ymax = upper),
                  fill = COL_CI, alpha = 0.18, inherit.aes = FALSE) } +
  { if (!is.null(ci_spot_band))
      geom_ribbon(data = ci_spot_band, aes(x = fpr, ymin = lower, ymax = upper),
                  fill = COL_CI, alpha = 0.18, inherit.aes = FALSE) } +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed",
              color = COL_DIAG, linewidth = 1.0) +
  geom_step(data = df_24,   aes(x = fpr, y = tpr), color = COL_24HR, linewidth = 2.0) +
  geom_step(data = df_spot, aes(x = fpr, y = tpr), color = COL_SPOT, linewidth = 2.0) +
  annotate("text", x = 0.56, y = 0.30, label = label_24hr,
           color = COL_24HR, size = 5.5, hjust = 0, fontface = "bold", lineheight = 1.25) +
  annotate("text", x = 0.56, y = 0.14, label = label_spot,
           color = COL_SPOT, size = 5.5, hjust = 0, fontface = "bold", lineheight = 1.25) +
  annotate("text", x = 0.98, y = 0.07,
           label = paste("DeLong", p_fmt(dl_aim2$p)),
           color = COL_TEXT, size = 5.2, hjust = 1, fontface = "italic") +
  annotate("text", x = 0.98, y = 0.01,
           label = paste0("n = ", nrow(df_c)),
           color = "grey50", size = 4.8, hjust = 1) +
  scale_x_continuous(name = "False Positive Rate", limits = c(0, 1),
                     breaks = c(0, 0.25, 0.5, 0.75, 1),
                     expand = expansion(mult = c(0.01, 0.02))) +
  scale_y_continuous(name = "Sensitivity", limits = c(0, 1),
                     breaks = c(0, 0.25, 0.5, 0.75, 1),
                     expand = expansion(mult = c(0.01, 0.02))) +
  labs(title = "ROC: Spot vs. 24-hr USG") +
  coord_cartesian(clip = "off") +
  theme_roc

ggsave("figures/Fig2_paired_ROC.png", p_roc, width = 8.5, height = 7, dpi = 300, bg = "white")
cat("Saved: figures/Fig2_paired_ROC.png\n")
p_roc

---
## Paper-ready summary

Run this cell last to print a formatted summary of all reported statistics.

In [ ]:
# ── Cell 11: Paper-ready summary ──────────────────────────────────────────────
cat(strrep("=", 72), "\n TABLE 1\n", strrep("=", 72), "\n")
cat(sprintf("YF n = %d  |  OF n = %d\n", nrow(young_demo), nrow(older_demo)))
table1_var("Age",            "Age (years)", young_demo, older_demo)
table1_var("Screening.BMI", "BMI (kg/m²)", young_demo, older_demo)

cat(strrep("=", 72), "\n AIM 1: AUC by marker and age group\n", strrep("=", 72), "\n")
cat(sprintf("%-22s  %-32s  %-32s  %s\n",
            "Marker", "YF: AUC (95% CI) [J]", "OF: AUC (95% CI) [J]", "DeLong p"))
cat(strrep("-", 100), "\n")

for (i in seq_along(markers)) {
  m   <- markers[i]; lab <- marker_labels[i]
  ry  <- safe_roc(young$AKI_binary, young[[m]])
  ro  <- safe_roc(older$AKI_binary, older[[m]])
  if (!is.null(ry) && !is.null(ro) && ry$direction != ro$direction)
    ro <- safe_roc(older$AKI_binary, older[[m]], direction = ry$direction)
  sy  <- roc_summary(ry); so <- roc_summary(ro)
  dl  <- delong_unpaired(ry, ro)
  fmt <- function(s) {
    if (is.null(s)) return("n/a")
    sprintf("%.2f (%.2f\u2013%.2f) [%.3f]", s$auc, s$ci_lo, s$ci_hi, s$J)
  }
  cat(sprintf("%-22s  %-32s  %-32s  %.4f\n",
              lab, fmt(sy), fmt(so), ifelse(is.na(dl$p), NA, dl$p)))
}

cat("\n", strrep("=", 72), "\n AIM 2: Spot vs. 24-hr USG (paired DeLong)\n", strrep("=", 72), "\n")
cat(sprintf("Spot USG : AUC = %.2f (%.2f\u2013%.2f) | J = %.3f | threshold > %.4f | Sens = %.2f | Spec = %.2f\n",
            s_spot$auc, s_spot$ci_lo, s_spot$ci_hi,
            s_spot$J, s_spot$threshold, s_spot$sens, s_spot$spec))
cat(sprintf("24-hr USG: AUC = %.2f (%.2f\u2013%.2f) | J = %.3f | threshold > %.4f | Sens = %.2f | Spec = %.2f\n",
            s_24hr$auc, s_24hr$ci_lo, s_24hr$ci_hi,
            s_24hr$J, s_24hr$threshold, s_24hr$sens, s_24hr$spec))
cat(sprintf("Paired DeLong: Z = %.3f  p = %.4f\n", dl_aim2$z, dl_aim2$p))